In [ ]:
import numpy as np
import soundfile as sf
import librosa
import matplotlib.pyplot as plt

def spectral_subtract(mix_path, noise_path, out_path=None,
                      n_fft=4096, hop_length=None,
                      floor_db=-40):
    """
    mix_path:    audio with air + milling
    noise_path:  air-only audio
    alpha:       subtraction aggressiveness (1.0 - 2.0)
    floor_db:    how much residual noise to keep (avoid musical artifacts)
    """
    if hop_length is None:
        hop_length = n_fft // 4

    # Load
    mix, sr = librosa.load(mix_path, sr=None, mono=True)
    noise, sr2 = librosa.load(noise_path, sr=None, mono=True)
    if sr2 != sr:
        raise ValueError(f"Sample rates differ: mix={sr}, noise={sr2}")

    # Convert to mono if needed
    if mix.ndim > 1:
        mix = mix.mean(axis=1)
    if noise.ndim > 1:
        noise = noise.mean(axis=1)

    # STFT
    X = librosa.stft(mix, n_fft=n_fft, hop_length=hop_length)
    N = librosa.stft(noise, n_fft=n_fft, hop_length=hop_length)

    X_mag, X_phase = np.abs(X), np.angle(X)
    N_mag = np.abs(N)

    # Estimate noise magnitude per frequency bin (median is robust)
    noise_profile = np.percentile(N_mag, 20, axis=1, keepdims=True)

    scale = np.sqrt(np.mean(X_mag**2) / (np.mean(N_mag**2) + 1e-9))
    noise_profile = noise_profile * scale

    # Subtract
    snr = (X_mag**2) / (noise_profile**2 + 1e-9)
    gain = snr / (snr + 1.0)
    clean_mag = X_mag * gain

    # Apply a floor
    floor = librosa.db_to_amplitude(floor_db)
    clean_mag = np.maximum(clean_mag, floor * np.max(X_mag))

    # Reconstruct using original phase
    Y = clean_mag * np.exp(1j * X_phase)
    y_clean = librosa.istft(Y, hop_length=hop_length)

    # Save if requested
    if out_path:
        sf.write(out_path, y_clean, sr)

    return y_clean, sr, X_mag, clean_mag

In [ ]:
import pywt

# ---- CWT params (tune these) ----
CWT_WIN_S    = 1.0     # short window keeps it fast
CWT_OFFSET_S = 2.0     # seconds into the recording (pick a stable part)
CWT_FMIN     = 5_000
CWT_FMAX     = 50_000  # or 96_000 if you really want full Nyquist and pain
CWT_NFREQ    = 128
WAVELET      = "cmor1.5-1.0"

def cwt_power_db(y, sr, fmin, fmax, nfreq, wavelet=WAVELET):
    freqs = np.geomspace(fmin, fmax, nfreq)
    dt = 1.0 / sr
    cf = pywt.central_frequency(wavelet)
    scales = cf / (freqs * dt)

    coefs, _ = pywt.cwt(y, scales, wavelet, sampling_period=dt)
    power = np.abs(coefs)**2
    P_db = 10.0 * np.log10(power + 1e-18)
    return freqs, P_db

def pick_window(y, sr, offset_s, win_s):
    start = int(offset_s * sr)
    win   = int(win_s * sr)
    if start + win > len(y):
        start = max(0, len(y) - win)
    return y[start:start+win], start

# Pick same window for before/after
y_before_win, start_i = pick_window(d_primary, sr, CWT_OFFSET_S, CWT_WIN_S)
y_after_win,  _       = pick_window(cleaned,   sr, CWT_OFFSET_S, CWT_WIN_S)

freqs, P_before = cwt_power_db(y_before_win, sr, CWT_FMIN, min(CWT_FMAX, sr/2), CWT_NFREQ)
_,     P_after  = cwt_power_db(y_after_win,  sr, CWT_FMIN, min(CWT_FMAX, sr/2), CWT_NFREQ)

# Time axis for the window
tt = np.arange(P_before.shape[1]) / sr

# Use identical color scaling for fair comparison
vmin = min(np.percentile(P_before, 5), np.percentile(P_after, 5))
vmax = max(np.percentile(P_before, 99), np.percentile(P_after, 99))

# --- Plot before ---
plt.figure(figsize=(12,4))
plt.pcolormesh(tt, freqs, P_before, shading="auto", cmap="magma", vmin=vmin, vmax=vmax)
plt.yscale("log")
plt.ylim(freqs.min(), freqs.max())
plt.xlabel("Time [s] (window)")
plt.ylabel("Freq [Hz] (log)")
plt.title(f"CWT BEFORE (primary) | window @ {start_i/sr:.2f}s")
plt.colorbar(label="Power [dB]")
plt.tight_layout()

# --- Plot after ---
plt.figure(figsize=(12,4))
plt.pcolormesh(tt, freqs, P_after, shading="auto", cmap="magma", vmin=vmin, vmax=vmax)
plt.yscale("log")
plt.ylim(freqs.min(), freqs.max())
plt.xlabel("Time [s] (window)")
plt.ylabel("Freq [Hz] (log)")
plt.title("CWT AFTER (cleaned = primary - estimated noise)")
plt.colorbar(label="Power [dB]")
plt.tight_layout()

# --- Difference map (after - before) ---
P_diff = P_after - P_before
plt.figure(figsize=(12,4))
plt.pcolormesh(tt, freqs, P_diff, shading="auto", cmap="RdBu_r", vmin=-10, vmax=10)
plt.yscale("log")
plt.ylim(freqs.min(), freqs.max())
plt.xlabel("Time [s] (window)")
plt.ylabel("Freq [Hz] (log)")
plt.title("CWT DIFF (after - before) [dB] | blue=decrease, red=increase")
plt.colorbar(label="Δ dB")
plt.tight_layout()

# --- Tiny numeric summaries (useful to print) ---
mean_prof_before = P_before.mean(axis=1)   # mean over time, per freq
mean_prof_after  = P_after.mean(axis=1)

# average delta in your ANC band (or any band you care about)
band_lo, band_hi = ANC_BAND  # reuse your tuple
m = (freqs >= band_lo) & (freqs <= band_hi)
avg_delta_band_db = float(np.mean(mean_prof_after[m] - mean_prof_before[m]))

# "similarity" of mean profiles (1.0 means basically same shape)
def cosine(a, b):
    a = a - a.mean(); b = b - b.mean()
    return float(np.dot(a,b) / ((np.linalg.norm(a)+1e-12)*(np.linalg.norm(b)+1e-12)))

sim_mean_profiles = cosine(mean_prof_before, mean_prof_after)

print(f"[CWT] Avg ΔdB in ANC_BAND {ANC_BAND}: {avg_delta_band_db:.2f} dB (negative = energy reduced)")
print(f"[CWT] Cosine similarity of mean CWT profiles: {sim_mean_profiles:.4f} (closer to 1 = very similar)")

In [ ]:
mix_path = "4_014mm_front_down.flac"
noise_path = "4_014mm_front_up.flac"

In [ ]:
# Get stats

stats = {
    'duration_sec': len(y_clean) / sr,
    'mean': np.mean(y_clean),
    'std': np.std(y_clean),
    'min': np.min(y_clean),
    'max': np.max(y_clean),
    'rms': np.sqrt(np.mean(y_clean**2)),
    'zero_crossing_rate': np.mean(librosa.zero_crossings(y_clean)),
    'energy': np.sum(y_clean**2)
}

print("Cleaned Audio Stats:")
for key, value in stats.items():
    print(f"{key}: {value}")

In [ ]:
mix, _ = librosa.load(mix_path, sr=None)   # keep original sr
print("mix peak:", np.max(np.abs(mix)))
print("clean peak:", np.max(np.abs(y_clean)))

In [ ]:
y_clean, sr, X_mag, clean_mag = spectral_subtract(
    mix_path, noise_path,
    n_fft=4096,
    hop_length=512,
)

In [ ]:
# Plot before
plt.figure(figsize=(12,4))
plt.imshow(20*np.log10(X_mag + 1e-9), aspect='auto', origin='lower')
plt.title("Before (Air + Milling)")
plt.colorbar(label="dB")
plt.tight_layout()
plt.show()

In [ ]:
# Plot after
plt.figure(figsize=(12,4))
plt.imshow(20*np.log10(clean_mag + 1e-9), aspect='auto', origin='lower')
plt.title("After (Noise Reduced)")
plt.colorbar(label="dB")
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(14, 10))
        
# Waveform
ax1 = plt.subplot(3, 1, 1)
librosa.display.waveshow(y_clean, sr=sr, ax=ax1)
ax1.set_title(f'Waveform', fontsize=12, fontweight='bold')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude')
ax1.grid(True, alpha=0.3)
        
# Spectrogram
ax2 = plt.subplot(3, 1, 2)
D = librosa.stft(y_clean)
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
img2 = librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='hz', ax=ax2)
ax2.set_title(f'Spectrogram', fontsize=12, fontweight='bold')
ax2.set_ylabel('Frequency (Hz)')
fig.colorbar(img2, ax=ax2, format='%+2.0f dB')
        
# Mel Spectrogram
ax3 = plt.subplot(3, 1, 3)
mel_spec = librosa.feature.melspectrogram(y=y_clean, sr=sr)
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
img3 = librosa.display.specshow(mel_spec_db, sr=sr, x_axis='time', y_axis='mel', ax=ax3)
ax3.set_title(f'Mel Spectrogram', fontsize=12, fontweight='bold')
ax3.set_ylabel('Mel Frequency')
ax3.set_xlabel('Time (s)')
fig.colorbar(img3, ax=ax3, format='%+2.0f dB')
        
plt.tight_layout()

In [ ]:
mix, sr = librosa.load(mix_path, sr=None, mono=True)
air, sr2 = librosa.load(noise_path, sr=None, mono=True)
assert sr == sr2, f"SR mismatch: {sr} vs {sr2}"

n_fft = 4096
hop = 512
n_mels = 128

def mel_spec(y):
    return librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop, n_mels=n_mels, power=2.0
    )

S_mix = mel_spec(mix)
S_air = mel_spec(air)

S_mix_db = librosa.power_to_db(S_mix, ref=np.max)
S_air_db = librosa.power_to_db(S_air, ref=np.max)

# --- 3) PCEN (noise-robust spectrogram) ---
S_mix_pcen = librosa.pcen(S_mix)

# --- 4) "Difference" spectrogram (highlights what changes vs air) ---
# Use median air profile over time (more stable than frame-by-frame subtraction)
air_profile_db = np.median(S_air_db, axis=1, keepdims=True)
S_diff_db = S_mix_db - air_profile_db

# --- 5) Plot ---
plt.figure(figsize=(12,4))
librosa.display.specshow(S_mix_db, sr=sr, hop_length=hop, x_axis="time", y_axis="mel")
plt.title("Mix: Log-Mel Spectrogram (dB)")
plt.colorbar(format="%+0.1f dB")
plt.tight_layout()
plt.show()

plt.figure(figsize=(12,4))
librosa.display.specshow(S_mix_pcen, sr=sr, hop_length=hop, x_axis="time", y_axis="mel")
plt.title("Mix: PCEN Mel Spectrogram (best for strong steady noise)")
plt.colorbar()
plt.tight_layout()
plt.show()

plt.figure(figsize=(12,4))
librosa.display.specshow(S_diff_db, sr=sr, hop_length=hop, x_axis="time", y_axis="mel")
plt.title("Difference: Mix(dB) - Air Profile(dB)")
plt.colorbar(format="%+0.1f dB")
plt.tight_layout()
plt.show()